In [0]:

# Notebook 07: Retail CSV ingestion using PySpark
# Reading retail data from CSV files and storing it in Delta tables.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    LongType, DoubleType
)

# Step 1: Get the current catalog and set the project schema.
catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_name = "retail_lakehouse"
volume_name = "retail_landing"

# Step 2: Create a landing volume for the incoming files.
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS
    `{catalog}`.`{schema_name}`.`{volume_name}`
""")

landing_path = (
    f"/Volumes/{catalog}/{schema_name}/{volume_name}"
)

# Step 3: Prepare sample orders and products for testing.
# These records simulate data received from a retail system.

orders_data = [
    ("O2001", "2026-10-08", "C001", "P101", 2, 499.0, "Completed", "West"),
    ("O2002", "2026-10-08", "C002", "P102", 1, 999.0, "Completed", "South"),
    ("O2003", "2026-10-09", "C003", "P103", 3, 199.0, "Pending", "North"),
    ("O2004", "2026-10-09", None, "P104", 2, 299.0, "Completed", "East"),
    ("O2005", "2026-10-09", "C004", "P101", -1, 499.0, "Completed", "West")
]

order_columns = [
    "order_id", "order_date", "customer_id", "product_id",
    "quantity", "unit_price", "status", "region"
]

orders_df = spark.createDataFrame(orders_data, order_columns)

products_data = [
    ("P101", "Wireless Mouse", "Accessories", 499.0),
    ("P102", "Mechanical Keyboard", "Accessories", 999.0),
    ("P103", "USB-C Hub", "Electronics", 199.0),
    ("P104", "Laptop Stand", "Accessories", 299.0)
]

product_columns = [
    "product_id", "product_name", "category", "standard_price"
]

products_df = spark.createDataFrame(
    products_data, product_columns
)

# Step 4: Save the sample data as CSV files.
# Spark writes the output into directories containing part-files.

orders_path = f"{landing_path}/orders"
products_path = f"{landing_path}/products"

(
    orders_df.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(orders_path)
)

(
    products_df.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(products_path)
)

# Step 5: Define the schemas before reading the CSV files.
# This keeps the expected column types consistent.

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", LongType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("region", StringType(), True)
])

products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("standard_price", DoubleType(), True)
])

# Step 6: Read the CSV files into Spark DataFrames.

orders_raw = (
    spark.read
    .option("header", True)
    .schema(orders_schema)
    .csv(orders_path)
)

products_raw = (
    spark.read
    .option("header", True)
    .schema(products_schema)
    .csv(products_path)
)

print("Orders loaded:", orders_raw.count())
print("Products loaded:", products_raw.count())

# Step 7: Store the raw records in Bronze Delta tables.
# Overwrite is being used for this initial learning exercise only.

orders_table = f"{catalog}.{schema_name}.bronze_orders"
products_table = f"{catalog}.{schema_name}.bronze_products"

(
    orders_raw.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(orders_table)
)

(
    products_raw.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(products_table)
)

# Step 8: Check that the saved tables contain all source records.

saved_orders = spark.table(orders_table)
saved_products = spark.table(products_table)

assert saved_orders.count() == orders_raw.count()
assert saved_products.count() == products_raw.count()

print("File ingestion completed successfully.")

display(saved_orders)
display(saved_products)
